# ENSO events (ONI) — data exploration
Source file: `data/el-nino-southern-oscillation-enso-el-nino-and-la-nina-events.csv` (downloaded by hand from HDX).
Goal: understand the file, check its quality and decide what needs cleaning later. Nothing is dropped or changed in the data here.

In [1]:
# Core libraries for cleaning and exploring data
import pandas as pd
import numpy as np
import pyarrow
# Read the CSV file into a DataFrame.
df_raw = pd.read_csv("../data/el-nino-southern-oscillation-enso-el-nino-and-la-nina-events.csv")

### loading the data
- The file is a **CSV**, so `pd.read_csv` is the right reader.
- It loads **918 rows x 8 columns** in *long format*: one row = one month (Niño 3.4 sea surface temperature, its climatology, its anomaly, and the El Niño / La Niña label).
- No `DtypeWarning` this time, and no column needs `low_memory=False`.

In [2]:
#copy the data frame to a new variable to avoid changing the original data frame
df = df_raw.copy()
df.head()

,date,TOTAL,ClimAdjust,ANOM,ANOM_trimester,ANOM_trimester_round,phase_trimester,phase_event
0,1950-01-01 00:00:00,24.56,26.18,-1.62,-1.336667,-1.3,lanina,lanina
1,1950-02-01 00:00:00,25.07,26.39,-1.32,-1.166667,-1.2,lanina,lanina
2,1950-03-01 00:00:00,25.88,26.95,-1.07,-1.183333,-1.2,lanina,lanina
3,1950-04-01 00:00:00,26.29,27.39,-1.11,-1.073333,-1.1,lanina,lanina
4,1950-05-01 00:00:00,26.19,27.56,-1.37,-0.850000,-0.8,lanina,lanina


### first look at the rows
- The series starts in **1950-01**. The first rows are cold (anomaly `-1.62` to `-1.07`) and labelled `lanina`.
- `ANOM_trimester` in the first row (`-1.34`) is not the January anomaly (`-1.62`): it is a 3-month average (see below).
- `date` looks like a date but is read as text, and each value ends with `00:00:00`.

In [3]:
print("Shape:", df.shape)
df.tail(3)

Shape: (918, 8)


,date,TOTAL,ClimAdjust,ANOM,ANOM_trimester,ANOM_trimester_round,phase_trimester,phase_event
915,2026-04-01 00:00:00,28.13,27.71,0.42,0.976667,1.0,elnino,neutral
916,2026-05-01 00:00:00,28.84,27.83,1.01,NaN,NaN,neutral,neutral
917,2026-06-01 00:00:00,29.17,27.67,1.50,NaN,NaN,neutral,neutral


### size and ordering of the file
- Shape confirmed: **(918, 8)**.
- The last rows are **2026-04, 2026-05 and 2026-06**. The last two have **no `ANOM_trimester`** (`NaN`) but are still labelled `neutral` even though their monthly anomaly is `+1.01` and `+1.50`.
- A `neutral` label with no value behind it means *unknown*, not *neutral*.

In [4]:
# Column names
print(f"Column names: {df.columns.tolist()}")

Column names: ['date', 'TOTAL', 'ClimAdjust', 'ANOM', 'ANOM_trimester', 'ANOM_trimester_round', 'phase_trimester', 'phase_event']


### column names
| Column | Meaning |
|---|---|
| `date` | Month (first day of the month) |
| `TOTAL` | Niño 3.4 sea surface temperature, in °C |
| `ClimAdjust` | Climatological (normal) SST for that month, in °C |
| `ANOM` | Monthly anomaly = `TOTAL - ClimAdjust` |
| `ANOM_trimester` | 3-month mean of `ANOM` (the ONI value) |
| `ANOM_trimester_round` | `ANOM_trimester` rounded to 1 decimal |
| `phase_trimester` | `elnino` / `lanina` / `neutral` from the 3-month value alone |
| `phase_event` | Same label, but a phase only counts once it lasts long enough to be an event |

- The meaning of the last two columns is confirmed by the checks further down.

In [5]:
df_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  918 non-null    str    
 1   TOTAL                 918 non-null    float64
 2   ClimAdjust            918 non-null    float64
 3   ANOM                  918 non-null    float64
 4   ANOM_trimester        916 non-null    float64
 5   ANOM_trimester_round  916 non-null    float64
 6   phase_trimester       918 non-null    str    
 7   phase_event           918 non-null    str    
dtypes: float64(5), str(3)
memory usage: 86.1 KB


### data types and memory
- 5 `float64` columns and 3 text columns (`date`, `phase_trimester`, `phase_event`).
- Memory is only about **86 KB**. This file is small, so memory is not a problem, but the types are still wrong for `date` and the two phase columns.
- `ANOM_trimester` and `ANOM_trimester_round` show **916 non-null** values instead of 918.

In [6]:
# Null counts per column
print(df_raw.isna().sum())

# Which rows hold the missing values?
df_raw[df_raw["ANOM_trimester"].isna()]

date                    0
TOTAL                   0
ClimAdjust              0
ANOM                    0
ANOM_trimester          2
ANOM_trimester_round    2
phase_trimester         0
phase_event             0
dtype: int64


,date,TOTAL,ClimAdjust,ANOM,ANOM_trimester,ANOM_trimester_round,phase_trimester,phase_event
916,2026-05-01 00:00:00,28.84,27.83,1.01,NaN,NaN,neutral,neutral
917,2026-06-01 00:00:00,29.17,27.67,1.50,NaN,NaN,neutral,neutral


### missing values
- Only two columns have nulls: **`ANOM_trimester` (2)** and **`ANOM_trimester_round` (2)**, and they are the same two rows: **2026-05 and 2026-06**.
- This is expected: a 3-month mean needs the next two months, so the last two months cannot have one. Every other column is fully populated.

In [7]:
# Exact duplicate rows and duplicated dates
print("Duplicate rows :", df_raw.duplicated().sum())
print("Duplicate dates:", df_raw["date"].duplicated().sum())

# What do the date strings look like?
print(df_raw["date"].str.len().value_counts())
print(df_raw["date"].str[-8:].value_counts())

Duplicate rows : 0
Duplicate dates: 0
date
19    918
Name: count, dtype: int64
date
00:00:00    918
Name: count, dtype: int64


### duplicates and date format
- **No duplicate rows and no duplicate dates.**
- All 918 date strings have the same length (19 characters) and all end in `00:00:00`, so there is **no time-of-day information** and `pd.to_datetime` is safe.

In [8]:
# 1. date: text -> datetime, 2. phase columns: text -> category
df["date"] = pd.to_datetime(df["date"])
for col in ["phase_trimester", "phase_event"]:
    df[col] = df[col].astype("category")

print(df.dtypes)
print("Before:", round(df_raw.memory_usage(deep=True).sum() / 1024**2, 3), "MB")
print("After :", round(df.memory_usage(deep=True).sum() / 1024**2, 3), "MB")

date                    datetime64[us]
TOTAL                          float64
ClimAdjust                     float64
ANOM                           float64
ANOM_trimester                 float64
ANOM_trimester_round           float64
phase_trimester               category
phase_event                   category
dtype: object
Before: 0.084 MB
After : 0.044 MB


### fixing the data types
- `date` is now `datetime64`, and the two phase columns are `category` (3 labels each).
- Memory went from **0.084 MB to 0.044 MB (about half)**. No rows or values were removed.

In [9]:
# Is there exactly one row per month, with no gaps?
expected = pd.date_range(df["date"].min(), df["date"].max(), freq="MS")
print("Range   :", df["date"].min().date(), "->", df["date"].max().date())
print("Expected:", len(expected), " Found:", len(df))
print("Missing months:", expected.difference(df["date"]).tolist())
print("All on day 1  :", (df["date"].dt.day == 1).all())
print("Years covered :", df["date"].dt.year.nunique())

Range   : 1950-01-01 -> 2026-06-01
Expected: 918  Found: 918
Missing months: []
All on day 1  : True
Years covered : 77


### is the series complete?
- **918 months, from 1950-01 to 2026-06, with no gaps** (77 calendar years: 1950 to 2026).
- Every date is on day 1 of its month, so months can be joined with other monthly files without any adjustment.

In [10]:
# Summary stats for the numeric columns
df.describe().round(2)

,date,TOTAL,ClimAdjust,ANOM,ANOM_trimester,ANOM_trimester_round
count,918,918.00,918.00,918.00,916.00,916.00
mean,1988-03-16 18:29:01.176470,26.91,26.88,0.03,0.03,0.03
min,1950-01-01 00:00:00,24.25,26.01,-2.09,-2.03,-2.00
25%,1969-02-08 00:00:00,26.21,26.50,-0.56,-0.54,-0.50
50%,1988-03-16 12:00:00,26.98,26.73,-0.03,-0.04,0.00
75%,2007-04-23 12:00:00,27.59,27.29,0.54,0.55,0.50
max,2026-06-01 00:00:00,29.41,27.94,2.77,2.75,2.80
std,NaN,0.99,0.50,0.85,0.83,0.83


### numeric summary
- **`ANOM` runs from `-2.09` to `+2.77` °C**, with a mean close to zero (`0.03`) and a standard deviation of `0.85`.
- `ANOM_trimester` (916 values) runs from `-2.03` to `+2.75`; the rounded column from `-2.0` to `+2.8`.
- The quartiles are `-0.56` and `+0.54`, so about half of all months sit within roughly +-0.55 °C of zero.
- `ClimAdjust` ranges from `26.01` to `27.94`, which is the seasonal cycle of the normal SST.

In [ ]:
# ANOM should equal TOTAL - ClimAdjust
print("Max |ANOM - (TOTAL - ClimAdjust)|:", (df["ANOM"] - (df["TOTAL"] - df["ClimAdjust"])).abs().max().round(3))

# Does the climatology (ClimAdjust) change over the years for the same month?
jan = df[df["date"].dt.month == 1].set_index("date")["ClimAdjust"]
print("ClimAdjust distinct values:", df["ClimAdjust"].nunique())
print("January ClimAdjust, first / last year:", jan.iloc[0], "/", jan.iloc[-1])
print("Seasonal range of ClimAdjust in 2000:", df[df["date"].dt.year == 2000]["ClimAdjust"].agg(["min", "max"]).tolist())

Max |ANOM - (TOTAL - ClimAdjust)|: 0.01
ClimAdjust distinct values: 104
January ClimAdjust, first / last year: 26.18 / 26.38
Seasonal range of ClimAdjust in 2000: [26.56, 27.81]


### how `ANOM` and `ClimAdjust` are built
- `ANOM = TOTAL - ClimAdjust`, with a largest difference of `0.01` (just rounding).
- `ClimAdjust` has **104 distinct values**: it follows the seasons (`26.56` to `27.81` inside the year 2000) and it also **drifts slowly over the years** (January is `26.18` in 1950 and `26.38` in 2026). So the normal is a *moving* climatology, not one fixed value per calendar month.

In [12]:
# Is ANOM_trimester the forward-looking 3-month mean of ANOM?
fwd = df["ANOM"].rolling(3).mean().shift(-2)
print("Max |ANOM_trimester - forward 3-month mean|:", (df["ANOM_trimester"] - fwd).abs().max().round(4))

# Is the rounded column just ANOM_trimester rounded to 1 decimal?
rdiff = (df["ANOM_trimester_round"] - df["ANOM_trimester"].round(1)).abs()
print("Rows where the rounded column differs from .round(1):", (rdiff > 0.001).sum(), " (max difference", rdiff.max().round(2), ")")
print(df.loc[rdiff > 0.001, ["date", "ANOM_trimester", "ANOM_trimester_round"]].to_string(index=False))

# Last months of the series
df.tail(4)[["date", "ANOM", "ANOM_trimester", "ANOM_trimester_round", "phase_trimester", "phase_event"]]

Max |ANOM_trimester - forward 3-month mean|: 0.0
Rows where the rounded column differs from .round(1): 4  (max difference 0.1 )
      date  ANOM_trimester  ANOM_trimester_round
1960-11-01            0.05                   0.1
1989-11-01           -0.05                  -0.1
1992-09-01           -0.25                  -0.3
1998-09-01           -1.35                  -1.3


,date,ANOM,ANOM_trimester,ANOM_trimester_round,phase_trimester,phase_event
914,2026-03-01,0.09,0.506667,0.5,elnino,neutral
915,2026-04-01,0.42,0.976667,1.0,elnino,neutral
916,2026-05-01,1.01,NaN,NaN,neutral,neutral
917,2026-06-01,1.50,NaN,NaN,neutral,neutral


### the 3-month mean and the rounded value
- `ANOM_trimester` equals the **forward-looking** 3-month mean of `ANOM` exactly (largest difference `0.0`): a row holds the mean of that month and the next two, so the January row is JFM.
- `ANOM_trimester_round` is that value rounded to 1 decimal, except in **4 half-way cases** (1960-11, 1989-11, 1992-09, 1998-09) where it differs by `0.1`. This is only a rounding-direction detail.
- The tail confirms the end of the file: 2026-03 and 2026-04 have values (`0.5`, `1.0`), the last two months do not.

In [13]:
# Phase counts and how the two phase columns relate
print(df["phase_trimester"].value_counts())
print(df["phase_event"].value_counts())
print(pd.crosstab(df["phase_trimester"], df["phase_event"]))

# Does phase_trimester follow the +-0.5 threshold on the rounded value?
chk = df.dropna(subset=["ANOM_trimester_round"])
print("elnino with value < 0.5 :", ((chk["phase_trimester"] == "elnino") & (chk["ANOM_trimester_round"] < 0.5)).sum())
print("lanina with value > -0.5:", ((chk["phase_trimester"] == "lanina") & (chk["ANOM_trimester_round"] > -0.5)).sum())
print("neutral outside +-0.5   :", ((chk["phase_trimester"] == "neutral") & (chk["ANOM_trimester_round"].abs() >= 0.5)).sum())

# Months that are warm/cool by season but neutral as an event: how long are those spells?
short = df[(df["phase_trimester"] != "neutral") & (df["phase_event"] == "neutral")]
spell = (short.index.to_series().diff() != 1).cumsum()
spells = short.groupby(spell).agg(start=("date", "min"), months=("date", "size"), phase=("phase_trimester", "first"))
print("Spells that never became events:", len(spells), " months per spell:", spells["months"].value_counts().sort_index().to_dict())
print(spells.tail(3))

phase_trimester
neutral    394
lanina     265
elnino     259
Name: count, dtype: int64
phase_event
neutral    423
elnino     249
lanina     246
Name: count, dtype: int64
phase_event      elnino  lanina  neutral
phase_trimester                         
elnino              249       0       10
lanina                0     246       19
neutral               0       0      394
elnino with value < 0.5 : 0
lanina with value > -0.5: 0
neutral outside +-0.5   : 0
Spells that never became events: 12  months per spell: {1: 1, 2: 7, 3: 2, 4: 2}
        start  months   phase
10 1996-11-01       2  lanina
11 2025-09-01       3  lanina
12 2026-03-01       2  elnino


### phases and the +-0.5 rule
- `phase_trimester`: **neutral 394, La Niña 265, El Niño 259**. `phase_event`: **neutral 423, El Niño 249, La Niña 246**.
- The cross-table shows that the two columns **never contradict each other** (no El Niño in one and La Niña in the other). `phase_event` only turns some months back into `neutral`: **10 El Niño months and 19 La Niña months**.
- `phase_trimester` follows the +-0.5 threshold on the rounded value exactly (all three checks return `0` exceptions).
- The 29 months that lose their label belong to **12 short spells of 1 to 4 months**, so they never reach the 5-month minimum of an event. The last spell (2026-03 and 2026-04, El Niño) is **not finished**: the file ends before it could reach 5 months.

In [14]:
# Turn runs of the same phase_event into events (a new event starts when the phase changes)
ev = df[["date", "ANOM_trimester_round", "phase_event"]].copy()
ev["run"] = (ev["phase_event"] != ev["phase_event"].shift()).cumsum()
events = (ev[ev["phase_event"] != "neutral"]
          .groupby(["run", "phase_event"], observed=True)
          .agg(start=("date", "min"), end=("date", "max"), months=("date", "size"),
               peak=("ANOM_trimester_round", lambda s: s.loc[s.abs().idxmax()]))
          .reset_index(level="phase_event")
          .reset_index(drop=True))

print(events["phase_event"].value_counts())
print("Shortest event (months):", events["months"].min(), " Longest:", events["months"].max())
print(events.groupby("phase_event", observed=True)["months"].agg(["count", "mean", "max"]).round(1))

# The 5 strongest events by absolute peak
events.reindex(events["peak"].abs().sort_values(ascending=False).index).head(5)

phase_event
elnino     25
lanina     20
neutral     0
Name: count, dtype: int64
Shortest event (months): 5  Longest: 32
             count  mean  max
phase_event                  
elnino          25  10.0   20
lanina          20  12.3   32


,phase_event,start,end,months,peak
37,elnino,2014-09-01,2016-04-01,20,2.8
26,elnino,1997-04-01,1998-04-01,13,2.4
18,elnino,1982-03-01,1983-05-01,15,2.2
44,elnino,2023-04-01,2024-04-01,13,2.1
12,elnino,1972-04-01,1973-02-01,11,2.1


### events built from `phase_event`
- **45 events since 1950: 25 El Niño and 20 La Niña.** The shortest lasts **5 months** (so the 5-month rule holds) and the longest **32 months** (a La Niña).
- El Niño events last 10.0 months on average (maximum 20); La Niña events 12.3 months (maximum 32).
- The five strongest events are all El Niño: **2014-09 to 2016-04 (peak +2.8)**, 1997-04 to 1998-04 (+2.4), 1982-03 to 1983-05 (+2.2), 2023-04 to 2024-04 (+2.1) and 1972-04 to 1973-02 (+2.1).
- Dates are the **first month** of the 3-month season, and the peak is taken from the rounded value.

In [15]:
# Peak ONI -> intensity class (Weak 0.5-0.9, Moderate 1.0-1.4, Strong 1.5-1.9, Very strong >= 2.0)
bins = [0.5, 1.0, 1.5, 2.0, 10]
labels = ["Weak", "Moderate", "Strong", "Very strong"]
events["intensity"] = pd.cut(events["peak"].abs(), bins=bins, labels=labels, right=False)
print(events.groupby(["phase_event", "intensity"], observed=True).size().unstack(fill_value=0))

# The event that is open at the end of the file
events.tail(2)

intensity    Weak  Moderate  Strong  Very strong
phase_event                                     
elnino          9         6       4            6
lanina          5         8       6            1


,phase_event,start,end,months,peak,intensity
43,lanina,2021-08-01,2022-12-01,17,-1.0,Moderate
44,elnino,2023-04-01,2024-04-01,13,2.1,Very strong


### intensity classes
| Intensity | El Niño | La Niña |
|---|---|---|
| Weak (0.5 to 0.9) | 9 | 5 |
| Moderate (1.0 to 1.4) | 6 | 8 |
| Strong (1.5 to 1.9) | 4 | 6 |
| Very strong (>= 2.0) | 6 | 1 |

- El Niño events reach "very strong" much more often than La Niña events (6 against 1).
- The last closed event is the **2023-04 to 2024-04 El Niño (very strong)**. The 2026 warm spell is not in this list yet because it has not lasted 5 months in this file.

In [16]:
# Save an untouched Parquet copy of the raw file (checkpoint before any cleaning)
import os
os.makedirs("../data/parquet", exist_ok=True)
df_raw.to_parquet("../data/parquet/enso_events_raw.parquet")

back = pd.read_parquet("../data/parquet/enso_events_raw.parquet")
print("Round trip identical:", back.equals(df_raw))
print("CSV :", round(os.path.getsize("../data/el-nino-southern-oscillation-enso-el-nino-and-la-nina-events.csv") / 1024, 1), "KB")
print("Parquet:", round(os.path.getsize("../data/parquet/enso_events_raw.parquet") / 1024, 1), "KB")

Round trip identical: True
CSV : 66.8 KB
Parquet: 22.5 KB


### Parquet checkpoint
- The raw file is saved as `data/parquet/enso_events_raw.parquet` and reading it back returns an **identical** DataFrame.
- Size drops from **66.8 KB to 22.5 KB**. This is the untouched copy; cleaning is done later in SQL.

### Conclusion: coverage and data quality

| Check | Result |
|---|---|
| Rows x columns | 918 x 8 |
| Period | 1950-01 to 2026-06, monthly |
| Gaps / duplicates | none / none |
| Missing values | 2 rows (2026-05, 2026-06) in `ANOM_trimester` and `ANOM_trimester_round` |
| Events found | 45 (25 El Niño, 20 La Niña), each at least 5 months |

### Conclusion: decisions for the cleaning step

| Column / topic | Action |
|---|---|
| `date` | Convert to a real date (done here) |
| `phase_trimester`, `phase_event` | Keep both as `category`; use `phase_event` for events and `phase_trimester` for the season label |
| `ANOM_trimester` | Forward-looking: shift the date by one month (or label it with the centre month) before joining with monthly rainfall or prices |
| Last two rows (2026-05, 2026-06) | Treat the phase as **unknown**, not `neutral` |
| 2026-03 and 2026-04 | An El Niño spell that is still open and not yet an event |
| `ClimAdjust` | Moving climatology: do not mix this `ANOM` with the anomalies of the Niño file without checking the difference (see the Niño notebook) |